# CDE data quality checks: Larch Road Health Centre

Checks an information delivery (IFC4 model, COBie 2.4 workbook, document register) against the EIR, using the rules in `rules.csv` and the data dictionary.

**How to run:** Runtime → Run all. Set `REPO_URL` in the next cell to your GitHub repo, or upload the repo folder to `/content/cde-data-quality`.

**Results:** a table with one row per check (rule, object, pass/fail/warn/flag, message), a submission decision based on EIR Section 9, and an Excel and HTML report in `out/`.

**Layer 4 (free-text review):** the check only ever *flags* something for a person to decide. It never passes or fails anything. With no API key it uses a keyword check. With an Anthropic API key saved as a Colab secret named `ANTHROPIC_API_KEY`, and `USE_LLM = True`, it asks a model instead.

In [ ]:
# ---- Setup: installs on Colab only; does nothing elsewhere
import subprocess, sys, os
from pathlib import Path

REPO_URL = ""   # e.g. "https://github.com/yourname/cde-data-quality.git"
BASE = Path("/content/cde-data-quality") if "google.colab" in sys.modules else Path(os.environ.get("CDE_BASE", "."))

if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ifcopenshell", "ifctester", "anthropic"], check=False)
    if REPO_URL and not BASE.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(BASE)], check=True)
sys.path.insert(0, str(BASE / "tools"))
print("Working folder:", BASE.resolve())
if not (BASE / "rules.csv").exists():
    print("rules.csv not found. Set REPO_URL above, or upload the repo folder to", BASE)

In [ ]:
# ---- Settings: change these for a different delivery
CHECK_DATE = "2027-06-04"                                   # the submission date being checked
IFC_FILE = BASE / "sample" / "LRHC-BMC-ZZ-00-M3-W-000001.ifc"
COBIE_FILE = BASE / "sample" / "LRHC-BMC-ZZ-ZZ-IE-W-000001.xlsx"
REGISTER_FILE = BASE / "sample" / "document_register.csv"
IDS_FILE = BASE / "ids" / "LRHC_asset_requirements.ids"
OUT = BASE / "out"

USE_LLM = False                 # True = use a model for layer 4 if an API key is available
LLM_MODEL = "claude-haiku-5-5"  # a small, cheap model is enough for this

In [ ]:
# ---- Load everything
import re, time, json
import pandas as pd
import numpy as np
from cde_lib import (Project, load_ifc, load_cobie, load_register, cobie_values, ifc_values, code_of, class_code,
                     is_empty, applies, parse_date, to_number, TYPE_CHECKS, same, write_excel, write_html)

t0 = time.time()
P = Project(BASE, CHECK_DATE)
entities = sorted(set(P.asset_types.ifc_entity))
for label, loader in [("ifc", lambda: load_ifc(IFC_FILE, entities)), ("cobie", lambda: load_cobie(COBIE_FILE)),
                      ("register", lambda: load_register(REGISTER_FILE))]:
    try:
        value = loader()
        if label == "ifc":
            P.ifc, IFC_READER = value
        else:
            setattr(P, label, value)
    except Exception as e:  # a file that won't open is itself a finding (L1-01 to L1-03)
        P.load_errors[label] = str(e)
        print(f"Could not load {label}: {e}")
if P.ifc:
    print(f"IFC read with {IFC_READER}: {len(P.ifc['assets'])} assets, {len(P.ifc['spaces'])} spaces")
if P.cobie:
    print(f"COBie: {len(P.cobie.get('Component', []))} components, sheets: {', '.join(P.cobie)}")
if P.register is not None:
    print(f"Register: {len(P.register)} rows")
print(f"{len(P.rules)} rules, {len(P.dd)} data dictionary rows")

## How a rule is written

Each rule is one function, tagged with its rule ID from `rules.csv`. It returns a list of `check(...)` results, one per object it looked at. `ok=True` means it passed. If `ok=False`, the result takes the rule's severity from `rules.csv` (fail or warn). Layer 4 results are never pass or fail: they're `flag` or `no flag`.

Rules L1-04 to L1-09 are **generic**. They read the data dictionary, so a new requirement added there is checked without touching any code.

In [ ]:
RULES = {}

def rule(rule_id):
    def register(fn):
        RULES[rule_id] = fn
        return fn
    return register

def check(obj, ok, message="", source="", code="", field="", value=""):
    return {"object": obj, "ok": bool(ok), "message": "" if ok else message, "source": source,
            "asset_code": code or code_of(obj), "field": field, "value": "" if value is None else str(value)}

def dd_rows(dd_refs):
    """Data dictionary rows by ID list ('DD-020;DD-041') or by selector name."""
    selectors = {
        "ALL_REQUIRED_Y": lambda r: r.required == "Y",
        "ALL_REQUIRED_C": lambda r: r.required == "C",
        "ALL_WITH_PATTERN": lambda r: r.pattern != "",
        "ALL_WITH_ALLOWED": lambda r: r.allowed_values != "",
        "ALL_WITH_MIN": lambda r: r.min_value.startswith(">"),
        "ALL_TYPED": lambda r: r.data_type in TYPE_CHECKS,
        "ALL_BOTH_SOURCES": lambda r: not r.ifc_source.startswith("n/a") and r.table != "Register",
    }
    if dd_refs in selectors:
        return [r for r in P.dd.itertuples() if selectors[dd_refs](r)]
    ids = dd_refs.split(";")
    return [r for r in P.dd.itertuples() if r.dd_id in ids]

def values(row, ifc=True, cobie=True):
    """(source, object, asset_code, value) for one data dictionary row."""
    out = []
    if cobie:
        out += [("REG" if row.table == "Register" else "COBie", o, c, v) for o, c, v in cobie_values(P, row)]
    if ifc:
        out += [("IFC", o, c, v) for o, c, v in ifc_values(P, row)]
    return out

def sheet(name):
    return P.cobie.get(name, pd.DataFrame()) if P.cobie else pd.DataFrame()

def attribute(sheet_name, row_name, name):
    a = sheet("Attribute")
    v = a[(a.SheetName == sheet_name) & (a.RowName == row_name) & (a.Name == name)].Value
    return v.iloc[0] if len(v) else ""

def type_row(pr):
    return P.type_row_for_pr(class_code(pr))

def cobie_type_pr(type_name):
    t = sheet("Type")
    v = t[t.Name == type_name].Category
    return class_code(v.iloc[0]) if len(v) else ""

def asset_row(code, pr):
    """asset_types row for an asset: by Pr code, or by asset code if that has only one variant."""
    r = type_row(pr) if pr else None
    if r is not None:
        return r
    rows = P.asset_types[P.asset_types.asset_code == code]
    return rows.iloc[0] if len(rows) == 1 else None

def ifc_pr(obj):
    return next((c for c in obj.get("class_codes", []) if str(c).startswith("Pr_")), "")

COBIE_FORMAT_ONLY = {"DD-042"}                       # COBie holds an email; IFC holds the company name
IFC_PATTERN_ROWS = {"DD-010", "DD-020", "DD-021", "DD-024", "DD-030", "DD-040"}

## Layer 1: metadata vs data dictionary

In [ ]:
@rule("L1-01")
def ifc_is_ifc4():
    schema = P.ifc["schema"] if P.ifc else P.load_errors.get("ifc", "not loaded")
    return [check(IFC_FILE.name, schema == "IFC4", f"IFC schema is {schema}, not IFC4", "IFC", "-", "schema", schema)]

@rule("L1-02")
def cobie_opens():
    return [check(COBIE_FILE.name, P.cobie is not None, f"COBie file could not be opened: {P.load_errors.get('cobie')}", "COBie", "-")]

@rule("L1-03")
def cobie_sheets_present():
    needed = ["Contact", "Facility", "Floor", "Space", "Type", "Component", "System", "Job", "Document", "Attribute"]
    have = set(P.cobie or [])
    return [check(s, s in have, f"Missing COBie sheet: {s}", "COBie", "-", "sheet") for s in needed]

@rule("L1-04")
def required_fields_filled():
    out = []
    for row in dd_rows("ALL_REQUIRED_Y"):
        for src, obj, code, v in values(row):
            out.append(check(obj, not is_empty(v, row.data_type), f"{row.attribute} is empty", src, code, row.field, v))
    return out

def job_in_scope(type_name):
    code = code_of(type_name)
    if code not in "AHU|FD|FDM|TMV|NCU|EAC|LF".split("|"):
        return False
    return code != "LF" or attribute("Type", type_name, "EmergencyLighting") == "Y"

@rule("L1-05")
def conditional_fields_filled():
    out = []
    jobs = sheet("Job")
    for row in dd_rows("ALL_REQUIRED_C"):
        if row.entity == "Job" and row.field != "TypeName":       # job existence is L3-16
            for _, j in jobs.iterrows():
                if job_in_scope(j.TypeName):
                    v = j[row.field]
                    out.append(check(j.TypeName, not is_empty(v, row.data_type),
                                     f"{row.attribute} is required for {j.TypeName}", "COBie", code_of(j.TypeName), row.field, v))
        elif row.table == "Register":
            for src, obj, code, v in values(row, ifc=False):
                out.append(check(obj, not is_empty(v), f"{row.attribute} is empty", src, "-", row.field, v))
    return out

@rule("L1-06")
def data_types_valid():
    out = []
    for row in dd_rows("ALL_TYPED"):
        test = TYPE_CHECKS[row.data_type]
        for src, obj, code, v in values(row, ifc=row.dd_id not in COBIE_FORMAT_ONLY):
            if not is_empty(v, row.data_type):
                out.append(check(obj, test(v), f"{row.attribute} is not a valid {row.data_type}: {v}", src, code, row.field, v))
    return out

@rule("L1-07")
def values_match_pattern():
    out = []
    for row in dd_rows("ALL_WITH_PATTERN"):
        pat = re.compile(row.pattern)
        for src, obj, code, v in values(row, ifc=row.dd_id in IFC_PATTERN_ROWS):
            if not is_empty(v, row.data_type):
                out.append(check(obj, pat.match(str(v)), f"{row.attribute} does not match the required format: {v}",
                                 src, code, row.field, v))
    return out

@rule("L1-08")
def values_in_allowed_list():
    out = []
    for row in dd_rows("ALL_WITH_ALLOWED"):
        allowed = row.allowed_values.split("|")
        numeric = row.data_type in ("decimal", "integer")
        for src, obj, code, v in values(row):
            if is_empty(v, row.data_type):
                continue
            ok = any(same(v, a, row.data_type) for a in allowed) if numeric else str(v) in allowed
            out.append(check(obj, ok, f"{row.attribute} is {v}; allowed: {', '.join(allowed)}", src, code, row.field, v))
    return out

@rule("L1-09")
def numbers_above_minimum():
    out = []
    for row in dd_rows("ALL_WITH_MIN"):
        floor = float(row.min_value[1:])
        for src, obj, code, v in values(row):
            n = to_number(v)
            if n is not None:
                out.append(check(obj, n > floor, f"{row.attribute} must be greater than {floor:g}: {v}", src, code, row.field, v))
    return out

def name_field_problems(name):
    parts = str(name).split("-")
    if len(parts) != 7:
        return [f"has {len(parts)} fields separated by hyphens, needs 7"]
    lists = ["project", "originator", "volume", "level", "doc_type", "role"]
    probs = [f"{lst} '{v}' not allowed ({', '.join(P.code_list(lst))})"
             for lst, v in zip(lists, parts[:6]) if v not in P.code_list(lst)]
    if not re.fullmatch(r"\d{6}", parts[6]):
        probs.append(f"number '{parts[6]}' is not 6 digits")
    return probs

@rule("L1-10")
def container_name_fields_valid():
    out = []
    names = [("COBie", n) for n in sheet("Document").get("Name", [])]
    names += [("REG", n) for n in dict.fromkeys(P.register.container_name)] if P.register is not None else []
    for src, n in names:
        probs = name_field_problems(n)
        out.append(check(n, not probs, f"{n}: " + "; ".join(probs), src, "-", "Name", n))
    return out

@rule("L1-11")
def one_title_per_container():
    titles = P.register.groupby("container_name").title.unique()
    return [check(c, len(t) == 1, f"{c} is used for {len(t)} titles: {', '.join(t)}", "REG", "-", "title")
            for c, t in titles.items()]

def uniqueness(names, source, what):
    counts = pd.Series(list(names)).value_counts()
    return [check(n, c == 1, f"{what} name {n} is used {c} times", source, field="Name", value=n) for n, c in counts.items()]

@rule("L1-12")
def asset_names_unique():
    return (uniqueness(sheet("Component").Name, "COBie", "Asset")
            + uniqueness([a["name"] for a in P.ifc["assets"]], "IFC", "Asset"))

@rule("L1-13")
def space_names_unique():
    return (uniqueness(sheet("Space").Name, "COBie", "Space")
            + uniqueness([s["name"] for s in P.ifc["spaces"]], "IFC", "Space"))

@rule("L1-14")
def name_prefix_matches_type():
    out = []
    for _, c in sheet("Component").iterrows():
        pr = cobie_type_pr(c.TypeName)
        r = type_row(pr)
        expected = r.asset_code if r is not None else None
        out.append(check(c.Name, expected == code_of(c.Name),
                         f"named as {code_of(c.Name)} but Pr code {pr} " + (f"means {expected}" if expected else "is not a recognised asset type"),
                         "COBie", field="TypeName", value=pr))
    for a in P.ifc["assets"]:
        r = type_row(ifc_pr(a))
        expected = r.asset_code if r is not None else None
        out.append(check(a["name"], expected == code_of(a["name"]),
                         f"named as {code_of(a['name'])} but Pr code {ifc_pr(a)} " + (f"means {expected}" if expected else "is not a recognised asset type"),
                         "IFC", field="classification", value=ifc_pr(a)))
    return out

@rule("L1-15")
def pr_code_allowed_for_type():
    out = []
    items = [("COBie", t.Name, class_code(t.Category)) for t in sheet("Type").itertuples()]
    items += [("IFC", n, next((c for c in t["class_codes"] if str(c).startswith("Pr_")), "")) for n, t in P.ifc["types"].items()]
    for src, name, pr in items:
        allowed = P.asset_types[P.asset_types.asset_code == code_of(name)].uniclass_pr.tolist()
        out.append(check(name, pr in allowed, f"Pr code {pr}; expected {', '.join(allowed)}", src, field="Category", value=pr))
    return out

@rule("L1-16")
def no_withdrawn_codes():
    withdrawn = set(P.code_list("withdrawn_uniclass"))
    items = [("COBie", r.Name, class_code(r.Category)) for s in ("Type", "Space", "System")
             for r in sheet(s).drop_duplicates("Name").itertuples()]
    items += [("IFC", o["name"], c) for group in (P.ifc["types"].values(), P.ifc["spaces"], P.ifc["systems"])
              for o in group for c in o["class_codes"]]
    return [check(n, c not in withdrawn, f"uses withdrawn Uniclass code {c}", src, field="Category", value=c)
            for src, n, c in items]

def schedule(col):
    return dict(zip(P.spaces_ref.space_name, P.spaces_ref[col]))

@rule("L1-17")
def sl_code_matches_schedule():
    sl = schedule("uniclass_sl")
    out = [check(s.Name, class_code(s.Category) == sl[s.Name], f"SL code {class_code(s.Category)}; schedule says {sl[s.Name]}",
                 "COBie", field="Category", value=s.Category) for s in sheet("Space").itertuples() if s.Name in sl]
    out += [check(s["name"], (s["class_codes"] or [""])[0] == sl[s["name"]],
                  f"SL code {(s['class_codes'] or [''])[0]}; schedule says {sl[s['name']]}", "IFC", field="classification")
            for s in P.ifc["spaces"] if s["name"] in sl]
    return out

@rule("L1-18")
def room_name_matches_schedule():
    rn = schedule("room_name")
    norm = lambda x: " ".join(str(x or "").lower().split())
    out = [check(s.Name, norm(s.Description) == norm(rn[s.Name]), f"room name '{s.Description}'; schedule says '{rn[s.Name]}'",
                 "COBie", field="Description", value=s.Description) for s in sheet("Space").itertuples() if s.Name in rn]
    out += [check(s["name"], norm(s["longname"]) == norm(rn[s["name"]]), f"room name '{s['longname']}'; schedule says '{rn[s['name']]}'",
                  "IFC", field="LongName", value=s["longname"]) for s in P.ifc["spaces"] if s["name"] in rn]
    return out

@rule("L1-19")
def patient_area_matches_schedule():
    pa = schedule("patient_area")
    out = [check(n, attribute("Space", n, "PatientArea") == pa[n], f"PatientArea {attribute('Space', n, 'PatientArea')}; schedule says {pa[n]}",
                 "COBie", "-", "PatientArea") for n in sheet("Space").Name if n in pa]
    out += [check(s["name"], s["psets"].get("LRHC_AssetData", {}).get("PatientArea") == pa[s["name"]],
                  f"PatientArea {s['psets'].get('LRHC_AssetData', {}).get('PatientArea')}; schedule says {pa[s['name']]}", "IFC", "-", "PatientArea")
            for s in P.ifc["spaces"] if s["name"] in pa]
    return out

@rule("L1-20")
def floor_names_valid():
    floors = P.code_list("floor")
    items = ([("COBie", f, f) for f in sheet("Floor").Name]
             + [("COBie", s.Name, s.FloorName) for s in sheet("Space").itertuples()]
             + [("IFC", f, f) for f in P.ifc["storeys"]])
    return [check(n, v in floors, f"floor {v} is not a project floor ({', '.join(floors)})", src, "-", "Floor", v) for src, n, v in items]

@rule("L1-21")
def emergency_flag_matches_pr():
    out = []
    for t in sheet("Type").itertuples():
        if code_of(t.Name) == "LF":
            r, v = type_row(t.Category), attribute("Type", t.Name, "EmergencyLighting")
            exp = r.emergency if r is not None else "?"
            out.append(check(t.Name, v == exp, f"EmergencyLighting is {v} but its Pr code means {exp}", "COBie", "LF", "EmergencyLighting", v))
    for n, t in P.ifc["types"].items():
        if code_of(n) == "LF":
            r = type_row(next((c for c in t["class_codes"] if str(c).startswith("Pr_")), ""))
            v, exp = t["psets"].get("LRHC_AssetData", {}).get("EmergencyLighting"), r.emergency if r is not None else "?"
            out.append(check(n, v == exp, f"EmergencyLighting is {v} but its Pr code means {exp}", "IFC", "LF", "EmergencyLighting", v))
    return out

@rule("L1-22")
def maintenance_interval_within_max():
    out = []
    for j in sheet("Job").itertuples():
        f, factor = to_number(j.Frequency), P.code_value("frequency_unit", j.FrequencyUnit)
        r = asset_row(code_of(j.TypeName), cobie_type_pr(j.TypeName))
        if f is None or factor is None or r is None or not r.max_interval_months:
            continue                                   # blanks are caught by L1-05
        months, limit = f * factor, float(r.max_interval_months)
        out.append(check(j.TypeName, months <= limit + 1e-6, f"maintained every {months:g} months; maximum is {limit:g}",
                         "COBie", field="Frequency", value=f"{j.Frequency} {j.FrequencyUnit}"))
    return out

@rule("L1-23")
def install_date_not_in_future():
    limit = parse_date(P.check_date)
    out = []
    for c in sheet("Component").itertuples():
        d = parse_date(c.InstallationDate)
        if d:
            out.append(check(c.Name, d <= limit, f"installation date {c.InstallationDate[:10]} is after the submission date {P.check_date}",
                             "COBie", field="InstallationDate", value=c.InstallationDate))
    return out

def status_code(s):
    return "A" if re.fullmatch(r"A[0-7]", str(s)) else str(s)

@rule("L1-24")
def status_valid_not_s0():
    ok_codes = set(P.code_list("status")) - {"S0"}
    return [check(r.container_name, status_code(r.status) in ok_codes, f"status {r.status} is not allowed",
                  "REG", "-", "status", r.status) for r in P.register.itertuples()]

@rule("L1-25")
def revision_matches_status():
    out = []
    for r in P.register.itertuples():
        need = "P" if re.fullmatch(r"S[1-4]", r.status) else "C" if status_code(r.status) in ("A", "CR") else None
        if need:
            out.append(check(r.container_name, str(r.revision).startswith(need),
                             f"{r.status} needs a {need} revision, has {r.revision}", "REG", "-", "revision", r.revision))
    return out

## Layer 2: status vs programme

In [ ]:
def rank(status):
    return P.code_value("status", status_code(status))

def entries(container, up_to=None):
    reg = P.register[P.register.container_name == container]
    if up_to:
        reg = reg[reg.status_date <= up_to]
    return reg.sort_values("status_date", kind="stable")

@rule("L2-01")
def planned_container_delivered():
    return [check(m.container_name, len(entries(m.container_name, m.due_date)) > 0,
                  f"{m.exchange} due {m.due_date}: not in the register by then", "REG", "-", "midp", m.midp_id)
            for m in P.midp.itertuples() if m.due_date <= P.check_date]

@rule("L2-02")
def right_status_by_due_date():
    out = []
    for m in P.midp.itertuples():
        e = entries(m.container_name, m.due_date)
        if m.due_date > P.check_date or e.empty:
            continue
        latest = e.iloc[-1]
        out.append(check(m.container_name, rank(latest.status) >= rank(m.required_status),
                         f"{m.exchange} due {m.due_date}: was {latest.status}, {m.required_status} required",
                         "REG", "-", "status", latest.status))
    return out

@rule("L2-03")
def early_warning_before_due():
    out = []
    for m in P.midp.itertuples():
        if m.due_date <= P.check_date:
            continue
        days = int(np.busday_count(P.check_date, m.due_date))
        if days <= 10:
            e = entries(m.container_name, P.check_date)
            current = e.iloc[-1].status if len(e) else "none"
            ok = len(e) and rank(current) >= rank(m.required_status)
            out.append(check(m.container_name, ok, f"{m.exchange} due {m.due_date} ({days} working days): is {current}, needs {m.required_status}",
                             "REG", "-", "status", current))
    return out

def rev_key(rev):
    m = re.fullmatch(r"([PC])(\d{2})", str(rev))
    return (0 if m.group(1) == "P" else 1, int(m.group(2))) if m else None

@rule("L2-04")
def revisions_move_forward():
    out = []
    for c in dict.fromkeys(P.register.container_name):
        best = None
        for r in entries(c).itertuples():
            k = rev_key(r.revision)
            if k is None:
                continue
            if best is not None:
                out.append(check(c, k > best[0], f"revision {r.revision} on {r.status_date} follows {best[1]}", "REG", "-", "revision", r.revision))
            if best is None or k > best[0]:
                best = (k, r.revision)
    return out

@rule("L2-05")
def container_in_midp():
    planned = set(P.midp.container_name)
    return [check(c, c in planned, f"{c} is not in the MIDP", "REG", "-", "container_name", c)
            for c in dict.fromkeys(P.register.container_name)]

## Layer 3: content and consistency

In [ ]:
def unique_names(names):
    s = pd.Series(list(names))
    return set(s[~s.duplicated(keep=False)])

def cobie_components():
    return sheet("Component")

@rule("L3-01")
def ifc_asset_in_cobie():
    cob = set(cobie_components().Name)
    return [check(a["name"], a["name"] in cob, "in the IFC but not in COBie", "IFC+COBie") for a in P.ifc["assets"]]

@rule("L3-02")
def cobie_asset_in_ifc():
    ifc = {a["name"] for a in P.ifc["assets"]}
    return [check(n, n in ifc, "in COBie but not in the IFC", "IFC+COBie") for n in cobie_components().Name]

@rule("L3-03")
def guids_match():
    comp = cobie_components()
    both = unique_names(comp.Name) & unique_names(a["name"] for a in P.ifc["assets"])
    ifc_guid = {a["name"]: a["guid"] for a in P.ifc["assets"]}
    out = []
    for c in comp.itertuples():
        in_ifc = c.Name in ifc_guid
        if c.Name in both:
            out.append(check(c.Name, c.ExtIdentifier == ifc_guid[c.Name],
                             f"COBie GUID {c.ExtIdentifier} does not match IFC GlobalId {ifc_guid[c.Name]}", "IFC+COBie", field="ExtIdentifier", value=c.ExtIdentifier))
        elif in_ifc:
            out.append(check(c.Name, False, "can't match GUIDs: the name is used more than once", "IFC+COBie", field="ExtIdentifier"))
    return out

@rule("L3-04")
def asset_in_exactly_one_space():
    out = []
    spaces = set(sheet("Space").Name)
    for a in P.ifc["assets"]:
        n = len(a["spaces"])
        out.append(check(a["name"], n == 1, f"IFC: in {n} spaces" + (" (contained in the storey instead)" if a["in_storey"] and n == 0 else ""), "IFC", field="Space"))
    for c in cobie_components().itertuples():
        out.append(check(c.Name, c.Space in spaces, f"COBie: space '{c.Space}' not found in the Space sheet", "COBie", field="Space", value=c.Space))
    both = unique_names(cobie_components().Name) & unique_names(a["name"] for a in P.ifc["assets"])
    cspace = dict(zip(cobie_components().Name, cobie_components().Space))
    for a in P.ifc["assets"]:
        if a["name"] in both and len(a["spaces"]) == 1 and cspace[a["name"]] in spaces:
            out.append(check(a["name"], a["spaces"][0] == cspace[a["name"]], f"IFC says {a['spaces'][0]}, COBie says {cspace[a['name']]}", "IFC+COBie", field="Space"))
    return out

@rule("L3-05")
def spaces_match_schedule():
    ref, cob, ifc = set(P.spaces_ref.space_name), set(sheet("Space").Name), {s["name"] for s in P.ifc["spaces"]}
    out = []
    for n in sorted(ref | cob | ifc):
        problems = [p for p, bad in [("not in the schedule", n not in ref), ("missing from COBie", n not in cob),
                                     ("missing from the IFC", n not in ifc)] if bad]
        out.append(check(n, not problems, f"space {n} is " + " and ".join(problems), "IFC+COBie", "-"))
    return out

@rule("L3-06")
def space_on_a_floor():
    floors = set(sheet("Floor").Name)
    out = [check(s["name"], s["storey"] is not None, "IFC: space is not on a storey", "IFC", "-") for s in P.ifc["spaces"]]
    out += [check(s.Name, s.FloorName in floors, f"COBie: floor {s.FloorName} not in the Floor sheet", "COBie", "-", "FloorName", s.FloorName)
            for s in sheet("Space").itertuples()]
    return out

@rule("L3-07")
def asset_linked_to_type():
    types = set(sheet("Type").Name)
    out = [check(a["name"], a["type_name"] is not None, "IFC: no type object", "IFC") for a in P.ifc["assets"]]
    out += [check(c.Name, c.TypeName in types, f"COBie: type {c.TypeName} not in the Type sheet", "COBie", field="TypeName", value=c.TypeName)
            for c in cobie_components().itertuples()]
    return out

@rule("L3-08")
def manufacturer_in_contact():
    emails = set(sheet("Contact").Email)
    return [check(t.Name, t.Manufacturer in emails, f"manufacturer {t.Manufacturer} is not in the Contact sheet", "COBie",
                  field="Manufacturer", value=t.Manufacturer) for t in sheet("Type").itertuples()]

@rule("L3-09")
def ifc_manufacturer_matches_contact():
    company = dict(zip(sheet("Contact").Email, sheet("Contact").Company))
    out = []
    for t in sheet("Type").itertuples():
        it = P.ifc["types"].get(t.Name)
        if it and t.Manufacturer in company:
            v = it["psets"].get("Pset_ManufacturerTypeInformation", {}).get("Manufacturer")
            out.append(check(t.Name, v == company[t.Manufacturer], f"IFC says {v}; COBie contact company is {company[t.Manufacturer]}",
                             "IFC+COBie", field="Manufacturer", value=v))
    return out

L3_10_SKIP = {"DD-024", "DD-023", "DD-042", "DD-010", "DD-020", "DD-040", "DD-001", "DD-080"}

@rule("L3-10")
def ifc_and_cobie_values_agree():
    out = []
    for row in dd_rows("ALL_BOTH_SOURCES"):
        if row.dd_id in L3_10_SKIP:
            continue
        ifc = [(o, v) for o, c, v in ifc_values(P, row)]
        cob = [(o, v) for o, c, v in cobie_values(P, row)]
        if not ifc or not cob:
            continue
        common = unique_names(o for o, _ in ifc) & unique_names(o for o, _ in cob)
        iv, cv = dict(ifc), dict(cob)
        for o in sorted(common):
            a, b = iv[o], cv[o]
            if is_empty(a, row.data_type) and is_empty(b, row.data_type):
                continue
            out.append(check(o, same(a, b, row.data_type), f"{row.attribute}: IFC '{a}', COBie '{b}'", "IFC+COBie", field=row.field, value=b))
    return out

@rule("L3-11")
def warranty_after_installation():
    out = []
    for c in cobie_components().itertuples():
        i, w = parse_date(c.InstallationDate), parse_date(c.WarrantyStartDate)
        if i and w:
            out.append(check(c.Name, w >= i, f"warranty starts {w}, before installation on {i}", "COBie", field="WarrantyStartDate", value=c.WarrantyStartDate))
    return out

@rule("L3-12")
def assets_in_right_system():
    out = []
    sysdf = sheet("System")
    ifc_sys = {s["name"]: s["class_codes"] for s in P.ifc["systems"]}
    for c in cobie_components().drop_duplicates("Name").itertuples():
        r = asset_row(code_of(c.Name), cobie_type_pr(c.TypeName))
        if r is None or not r.uniclass_ss:
            continue
        rows = sysdf[sysdf.ComponentNames == c.Name]
        matching = {s.Name for s in rows.itertuples() if class_code(s.Category) == r.uniclass_ss}
        out.append(check(c.Name, len(matching) == 1 and rows.Name.nunique() == 1,
                         f"COBie: in {rows.Name.nunique()} systems, {len(matching)} with code {r.uniclass_ss}", "COBie", field="System"))
    for a in P.ifc["assets"]:
        r = asset_row(code_of(a["name"]), ifc_pr(a))
        if r is None or not r.uniclass_ss:
            continue
        matching = [s for s in a["systems"] if r.uniclass_ss in ifc_sys.get(s, [])]
        out.append(check(a["name"], len(matching) == 1 and len(a["systems"]) == 1,
                         f"IFC: in {len(a['systems'])} systems, {len(matching)} with code {r.uniclass_ss}", "IFC", field="System"))
    return out

@rule("L3-13")
def patient_basins_have_tmv():
    comp = cobie_components()
    space_of = dict(zip(comp.Name, comp.Space))
    patient = {n for n in sheet("Space").Name if attribute("Space", n, "PatientArea") == "Y"}
    serves = {t: attribute("Component", t, "ServesComponent") for t in comp.Name if code_of(t) == "TMV"}
    served = pd.Series(list(serves.values())).value_counts()
    out = []
    for w in comp[comp.Name.str.startswith("WHB-") & comp.Space.isin(patient)].Name:
        n = int(served.get(w, 0))
        out.append(check(w, n == 1, f"basin in patient area {space_of[w]} is served by {n} TMVs (needs exactly 1)", "COBie", "WHB", "ServesComponent"))
    for t, w in serves.items():
        out.append(check(t, w in space_of and space_of[w] == space_of[t],
                         f"serves {w} in {space_of.get(w, 'unknown space')}, but the TMV is in {space_of[t]}", "COBie", "TMV", "ServesComponent", w))
    return out

def spaces_needing(clause):
    return P.spaces_ref[P.spaces_ref.eir_clause.str.contains(clause)]

@rule("L3-14")
def call_points_where_required():
    comp = cobie_components()
    out = []
    for s in spaces_needing("LOC-04").itertuples():
        need = "EAC" if s.uniclass_sl == "SL_35_80_03" else "NCU"
        have = (comp[comp.Space == s.space_name].Name.map(code_of) == need).any()
        out.append(check(s.space_name, have, f"{s.room_name} has no {need}", "COBie", need, "Space"))
    return out

@rule("L3-15")
def emergency_lighting_where_required():
    comp = cobie_components()
    out = []
    for s in spaces_needing("LOC-05").itertuples():
        lfs = comp[(comp.Space == s.space_name) & comp.Name.str.startswith("LF-")]
        have = any(attribute("Type", t, "EmergencyLighting") == "Y" for t in lfs.TypeName)
        out.append(check(s.space_name, have, f"{s.room_name} has no emergency light fitting", "COBie", "LF", "Space"))
    return out

@rule("L3-16")
def ppm_job_for_each_type():
    jobs = sheet("Job")
    ppm = set(jobs[jobs.Category == "PPM"].TypeName)
    return [check(t, t in ppm, "no PPM job in the Job sheet", "COBie", field="Job") for t in sheet("Type").Name if job_in_scope(t)]

@rule("L3-17")
def asset_count_as_expected():
    out = []
    expected = {(r.asset_code, r.variant): int(r.expected_qty) for r in P.asset_types.itertuples()}
    comp = cobie_components()
    cob_rows = [asset_row(code_of(c.Name), cobie_type_pr(c.TypeName)) for c in comp.itertuples()]
    ifc_rows = [asset_row(code_of(a["name"]), ifc_pr(a)) for a in P.ifc["assets"]]
    for src, rows in [("COBie", cob_rows), ("IFC", ifc_rows)]:
        found = pd.Series([(r.asset_code, r.variant) for r in rows if r is not None]).value_counts()
        for (code, variant), qty in expected.items():
            n = int(found.get((code, variant), 0))
            label = f"{code} {variant}".strip()
            out.append(check(label, n == qty, f"{label}: {n} found, {qty} expected", src, code, "count", n))
    return out

@rule("L3-18")
def ifc_entity_matches_type():
    out = []
    for a in P.ifc["assets"]:
        r = asset_row(code_of(a["name"]), ifc_pr(a))
        if r is None:
            continue
        t = P.ifc["types"].get(a["type_name"]) or {}
        predef = a["predefined"] if a["predefined"] not in (None, "NOTDEFINED") else t.get("predefined")
        otype = a["object_type"] or t.get("element_type")
        probs = []
        if a["entity"].upper() != r.ifc_entity.upper():
            probs.append(f"entity {a['entity']} (expected {r.ifc_entity})")
        if r.ifc_predefined_type and predef != r.ifc_predefined_type:
            probs.append(f"predefined type {predef} (expected {r.ifc_predefined_type})")
        if r.ifc_object_type and otype != r.ifc_object_type:
            probs.append(f"object type {otype} (expected {r.ifc_object_type})")
        out.append(check(a["name"], not probs, "; ".join(probs), "IFC", field="entity", value=a["entity"]))
    return out

## Layer 4: free-text review (flags only, a person decides)

Each job description is checked for the tasks the EIR expects (AIR-28). The result is `flag` or `no flag`, never pass or fail.

- **Keyword method (default):** looks for the required words. Fast, free, and nothing leaves this notebook. It misses tasks described in other words.
- **Model method:** sends the description (not the rest of the file) to a model and asks whether the tasks are covered. Needs an API key. That means paying the provider and sending client data outside the organisation, so it's the client's call.

In [ ]:
L4_REQUIREMENTS = {   # what each check looks for: label -> keywords that count as covering it
    "L4-02": {"outlet temperature check": ["temperature"], "fail-safe test": ["fail-safe", "failsafe", "isolat"]},
    "L4-03": {"drop test": ["drop test", "release"], "reset": ["reset"]},
}
GENERIC = re.compile(r"as per (the )?manufacturer|in accordance with (the )?manufacturer|as required|routine service", re.I)

def llm_client():
    if not USE_LLM:
        return None
    key = os.environ.get("ANTHROPIC_API_KEY")
    if not key and "google.colab" in sys.modules:
        try:
            from google.colab import userdata
            key = userdata.get("ANTHROPIC_API_KEY")
        except Exception:
            key = None
    if not key:
        print("USE_LLM is on but no ANTHROPIC_API_KEY found: using the keyword method")
        return None
    import anthropic
    return anthropic.Anthropic(api_key=key)

LLM = llm_client()

def missing_tasks(description, required):
    """Return the required tasks the description doesn't appear to cover."""
    if LLM:
        prompt = (f"A maintenance task description for a building asset reads:\n\"{description}\"\n\n"
                  f"Does it clearly include each of these tasks: {', '.join(required)}? "
                  'Reply with JSON only: {"missing": [tasks not covered], "reason": "one sentence"}')
        try:
            msg = LLM.messages.create(model=LLM_MODEL, max_tokens=200, messages=[{"role": "user", "content": prompt}])
            text = msg.content[0].text
            return json.loads(text[text.index("{"): text.rindex("}") + 1])["missing"], "model"
        except Exception as e:
            print("Model call failed, using keywords:", e)
    d = description.lower()
    return [label for label, words in required.items() if not any(w in d for w in words)], "keywords"

def jobs_for(codes):
    j = sheet("Job")
    return [r for r in j.itertuples() if code_of(r.TypeName) in codes and job_in_scope(r.TypeName) and not is_empty(r.Description)]

@rule("L4-01")
def task_description_is_specific():
    out = []
    for j in jobs_for("AHU|FD|FDM|TMV|NCU|EAC|LF".split("|")):
        generic = bool(GENERIC.search(j.Description)) or len(j.Description.split()) < 6
        if LLM:
            missing, how = missing_tasks(j.Description, {"specific named maintenance tasks (not just 'follow the manufacturer')": []})
            generic = bool(missing)
        out.append(check(j.TypeName, not generic, f"task description may be too generic: '{j.Description}'", "COBie", field="Description", value=j.Description))
    return out

@rule("L4-02")
def tmv_tasks_complete():
    out = []
    for j in jobs_for(["TMV"]):
        missing, how = missing_tasks(j.Description, L4_REQUIREMENTS["L4-02"])
        out.append(check(j.TypeName, not missing, f"may be missing: {', '.join(missing)} ({how})", "COBie", field="Description", value=j.Description))
    return out

@rule("L4-03")
def damper_tasks_complete():
    out = []
    for j in jobs_for(["FDM"]):
        missing, how = missing_tasks(j.Description, L4_REQUIREMENTS["L4-03"])
        out.append(check(j.TypeName, not missing, f"may be missing: {', '.join(missing)} ({how})", "COBie", field="Description", value=j.Description))
    return out

## Run every rule

In [ ]:
t_run = time.time()                              # times the rule run only, not the time spent between cells
rows, timings = [], []
for r in P.rules.itertuples():
    if r.layer == "D":
        continue
    fn = RULES.get(r.rule_id)
    start = time.time()
    if fn is None:
        found = [check("-", False, "rule not implemented yet")]
        found[0]["status"] = "error"
    else:
        try:
            found = fn()
        except Exception as e:                       # a broken rule shouldn't stop the run
            found = [check("-", False, f"rule raised an error: {type(e).__name__}: {e}")]
            found[0]["status"] = "error"
    timings.append((r.rule_id, time.time() - start, len(found)))
    for f in found:
        if "status" not in f:
            if r.layer == "4":
                f["status"] = "no flag" if f["ok"] else "flag"
            else:
                f["status"] = "pass" if f["ok"] else r.severity
        rows.append(dict(rule_id=r.rule_id, layer=r.layer, eir_clause=r.eir_clause, rule=r.name, **f))

results = pd.DataFrame(rows).drop(columns="ok")

COMP_TYPE = dict(zip(cobie_components().Name, cobie_components().TypeName))
IFC_ASSET_PR = {a["name"]: ifc_pr(a) for a in P.ifc["assets"]}
IFC_TYPE_PR = {n: next((c for c in t["class_codes"] if str(c).startswith("Pr_")), "") for n, t in P.ifc["types"].items()}

def safety_critical(row):
    """Y/N from the object's own Pr code, read from the file the result came from."""
    code, obj = row.asset_code, row.object
    if not code or code == "-":
        return ""
    if row.source == "IFC":
        pr = IFC_ASSET_PR.get(obj) or IFC_TYPE_PR.get(obj, "")
    else:
        pr = cobie_type_pr(obj) or cobie_type_pr(COMP_TYPE.get(obj, ""))
    return "Y" if P.is_safety_critical(code, pr or None) else "N"

results["safety_critical"] = results.apply(safety_critical, axis=1)
results = results[["rule_id", "layer", "eir_clause", "rule", "source", "object", "asset_code", "safety_critical",
                   "field", "value", "status", "message"]]
RUN_SECONDS = time.time() - t_run
print(f"{len(results)} checks in {RUN_SECONDS:.1f} s")
print(results.status.value_counts().to_string())

## Submission decision (EIR Section 9)

In [ ]:
fails = results[results.status == "fail"]
def any_fail(rule_ids):
    return fails[fails.rule_id.isin(rule_ids)]

critical_fails = pd.concat([fails[fails.safety_critical == "Y"], any_fail(["L1-19", "L3-13", "L3-14", "L3-15"])]).drop_duplicates()
asset_checks = results[results.layer.isin(["1", "3"]) & (results.safety_critical == "N") & results.status.isin(["pass", "fail"])]
rate = 100 * (asset_checks.status == "fail").mean() if len(asset_checks) else 0.0
container_rules = ["L1-10", "L1-11", "L1-24", "L1-25", "L2-01", "L2-02", "L2-04"]
rejected_containers = sorted(set(any_fail(container_rules).object))

decisions = pd.DataFrame([
    ("D-01", "ACC-01", "Files usable", len(any_fail(["L1-01", "L1-02", "L1-03"])), "Reject submission" if len(any_fail(["L1-01", "L1-02", "L1-03"])) else "OK"),
    ("D-02", "ACC-02", "IFC and COBie match", len(any_fail(["L3-01", "L3-02", "L3-03"])), "Reject submission" if len(any_fail(["L3-01", "L3-02", "L3-03"])) else "OK"),
    ("D-03", "ACC-03", "Safety-critical fails", len(critical_fails), "Reject submission" if len(critical_fails) else "OK"),
    ("D-04", "ACC-04", "Containers rejected", len(rejected_containers), ", ".join(rejected_containers) or "None"),
    ("D-05", "ACC-05/06", f"Non-critical fail rate {rate:.1f}% of {len(asset_checks)} checks", round(rate, 1),
     "Reject submission" if rate > 5 else "Accept with corrections" if rate > 0 else "OK"),
    ("D-06", "ACC-07/08", "Warnings to answer / flags to review", f"{(results.status == 'warn').sum()} / {(results.status == 'flag').sum()}",
     "Response list and reviewer queue"),
], columns=["rule_id", "eir_clause", "test", "count", "outcome"])
OUTCOME = "REJECTED" if decisions.outcome.str.startswith("Reject submission").any() else \
          "ACCEPTED WITH CORRECTIONS" if (results.status == "fail").any() else "ACCEPTED"
print("Submission outcome:", OUTCOME)
decisions

## Summaries

In [ ]:
pivot = lambda by: (results.pivot_table(index=by, columns="status", values="object", aggfunc="count", fill_value=0)
                    .reindex(columns=["pass", "fail", "warn", "flag", "no flag", "error"], fill_value=0))
by_rule = pivot(["rule_id", "rule"]).reset_index()
by_asset = pivot(["asset_code"]).reset_index()
by_asset = by_asset[~by_asset.asset_code.isin(["", "-"])]
issues = results[results.status.isin(["fail", "warn", "flag", "error"])].sort_values(["layer", "rule_id", "object"])
print(f"{len(issues)} issues: fail {(issues.status == 'fail').sum()}, warn {(issues.status == 'warn').sum()}, flag {(issues.status == 'flag').sum()}")
issues

## Write the reports

In [ ]:
OUT.mkdir(exist_ok=True)
summary = pd.DataFrame([("Outcome", OUTCOME), ("Check date", CHECK_DATE), ("IFC", IFC_FILE.name), ("COBie", COBIE_FILE.name),
                        ("Register", REGISTER_FILE.name), ("IFC reader", IFC_READER if P.ifc else "-"),
                        ("Layer 4 method", "model" if LLM else "keywords"), ("Checks run", len(results)),
                        ("Run time (s)", round(RUN_SECONDS, 1))], columns=["item", "value"])
sheets = {"Summary": summary, "Decision": decisions, "Issues": issues, "By rule": by_rule, "By asset type": by_asset,
          "All checks": results}
write_excel(OUT / "cde_check_report.xlsx", sheets)
write_html(OUT / "cde_check_report.html", "CDE data quality report: Larch Road Health Centre",
           [f"<b>Outcome: {OUTCOME}</b>", f"Check date {CHECK_DATE}", f"{len(results)} checks, {len(issues)} issues",
            f"Run time {RUN_SECONDS:.1f} s"],
           {"Decision": decisions, "Issues": issues, "By rule": by_rule, "By asset type": by_asset})
print("Reports written to", OUT)

## IDS check with ifctester (Colab)

Runs the IDS file against the IFC, as an independent second check of the IFC-side requirements. It also validates the IFC against the IFC4 schema. This needs ifcopenshell and ifctester, which the setup cell installs on Colab.

In [ ]:
try:
    import ifcopenshell, ifcopenshell.validate
    from ifctester import ids, reporter
    model = ifcopenshell.open(str(IFC_FILE))
    log = ifcopenshell.validate.json_logger()
    ifcopenshell.validate.validate(model, log)
    print(f"IFC schema validation: {len(log.statements)} issues")
    for s in log.statements[:10]:
        print(" -", s.get("message", s))
    spec = ids.open(str(IDS_FILE), validate=True)
    spec.validate(model)
    reporter.Console(spec).report()
    html = reporter.Html(spec)
    html.report()
    html.to_file(str(OUT / "ids_report.html"))
    print("IDS report written to", OUT / "ids_report.html")
except ImportError:
    print("ifcopenshell / ifctester not installed here: run this cell on Colab")
except Exception as e:
    print(f"IDS check could not run: {type(e).__name__}: {e}")

## Next

Stage 5 compares these results with `seeded_errors.csv` to score what was caught, missed and wrongly flagged.